# Worksheet 4, Parts 4–5 — Final abundances and the LaTeX table
### HE 1002-0755

In [ ]:
import os
import numpy as np
from astropy.table import Table
os.makedirs("results", exist_ok=True)

# >>> point this at your SMHR abundance-table export from the coadd session <<<
ABUND = "../part3_abundance_diagnostics/test_input/TEST_0543_abundance_table.txt"
IS_TEST = "TEST_" in os.path.basename(ABUND)
tab = Table.read(ABUND, format="ascii.fixed_width_two_line")
tab

## 1. Solar abundances and the check

In [ ]:
ASPLUND09 = {"Li": 1.05, "C": 8.43, "N": 7.83, "O": 8.69, "Na": 6.24, "Mg": 7.60, "Al": 6.45, "Si": 7.51,
    "K": 5.03, "Ca": 6.34, "Sc": 3.15, "Ti": 4.95, "V": 3.93, "Cr": 5.64, "Mn": 5.43, "Fe": 7.50, "Co": 4.99,
    "Ni": 6.22, "Cu": 4.19, "Zn": 4.56, "Sr": 2.87, "Y": 2.21, "Zr": 2.58, "Mo": 1.88, "Ru": 1.75, "Pd": 1.57,
    "Ba": 2.18, "La": 1.10, "Ce": 1.58, "Pr": 0.72, "Nd": 1.42, "Sm": 0.96, "Eu": 0.52, "Gd": 1.07, "Tb": 0.30,
    "Dy": 1.10, "Ho": 0.48, "Er": 0.92, "Tm": 0.10, "Yb": 0.84, "Lu": 0.10, "Hf": 0.85, "Os": 1.40, "Ir": 1.38,
    "Pb": 1.75, "Th": 0.02}
Z = {3: "Li", 6: "C", 7: "N", 8: "O", 11: "Na", 12: "Mg", 13: "Al", 14: "Si", 19: "K", 20: "Ca", 21: "Sc", 22: "Ti",
     23: "V", 24: "Cr", 25: "Mn", 26: "Fe", 27: "Co", 28: "Ni", 29: "Cu", 30: "Zn", 38: "Sr", 39: "Y", 40: "Zr",
     42: "Mo", 44: "Ru", 46: "Pd", 56: "Ba", 57: "La", 58: "Ce", 59: "Pr", 60: "Nd", 62: "Sm", 63: "Eu", 64: "Gd",
     65: "Tb", 66: "Dy", 67: "Ho", 68: "Er", 69: "Tm", 70: "Yb", 71: "Lu", 72: "Hf", 76: "Os", 77: "Ir", 82: "Pb", 90: "Th"}

el  = [Z[int(s)] for s in tab["species"]]
ion = [int(round(10*(s - int(s)))) + 1 for s in tab["species"]]       # 26.0 -> Fe I, 26.1 -> Fe II
sun = np.array([ASPLUND09[e] for e in el])
feh = float(tab["logeps"][np.isclose(tab["species"], 26.0)][0]) - ASPLUND09["Fe"]
XH  = tab["logeps"] - sun
XFe = XH - feh
print(f"[Fe/H] (Fe I) = {feh:+.2f}" + ("   [TEST INPUT]" if IS_TEST else ""))
print(f"max |[X/H] ours - SMHR|  = {np.max(np.abs(XH - tab['[X/H]'])):.3f}")
print(f"max |[X/Fe] ours - SMHR| = {np.max(np.abs(XFe - tab['[X/Fe]'])):.3f}   (export is rounded to 0.01)")

## 2. LaTeX table

In [ ]:
roman = {1: "I", 2: "II", 3: "III"}
order = np.lexsort((ion, [list(Z.values()).index(e) for e in el]))
lines = [r"\begin{deluxetable}{lcccccrr}",
         r"\tablecaption{LTE abundances of HE~1002$-$0755\label{tab:abundances}}",
         r"\tablehead{\colhead{Element} & \colhead{Ion} & \colhead{$\log\epsilon_\odot(\mathrm{X})$} & \colhead{$N$} &"
         r" \colhead{$\log\epsilon(\mathrm{X})$} & \colhead{$\sigma$} & \colhead{[X/H]} & \colhead{[X/Fe]}}",
         r"\startdata"]
prev = None
for i in order:
    name = el[i] if el[i] != prev else ""
    prev = el[i]
    lines.append(f"{name} & {roman[ion[i]]} & {sun[i]:.2f} & {int(tab['N'][i])} & {tab['logeps'][i]:.2f} & \\nodata & "
                 f"${XH[i]:+.2f}$ & ${XFe[i]:+.2f}$ \\\\")
lines[-1] = lines[-1].rstrip(" \\")
lines += [r"\enddata",
          r"\tablecomments{Solar abundances from \citet{asplund09}. [X/Fe] for Fe\,\textsc{ii} is relative to Fe\,\textsc{i}."
          r" Uncertainties $\sigma$ will be added in Worksheet 5.}",
          r"\end{deluxetable}"]
if IS_TEST:
    lines.insert(0, "% TEST INPUT: single exposure 2009-02-05, lines not checked - DO NOT USE IN THE PAPER")
out = "results/abundance_table" + ("_TEST" if IS_TEST else "") + ".tex"
open(out, "w").write("\n".join(lines) + "\n")
print(open(out).read())